# **Importing and downloading the needed libraries**

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import log_loss
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import BatchNormalization
import tensorflow as tf
from transformers import create_optimizer
from transformers import TFAutoModelForSequenceClassification
from transformers import AutoTokenizer
from sklearn.metrics import classification_report

In [ ]:
!pip install tensorflow

In [ ]:
!pip install transformers

# **loading the original data because transformers deals with raw data**

In [ ]:
df_trans = pd.read_csv("/content/altibbi_specialty_data.csv")

In [ ]:
df_trans = df_trans.drop("name_ar", axis=1)
df_trans = df_trans.sample(n=25000, random_state=42)

In [ ]:
x = df_trans["question_body"].astype(str).tolist()
y = df_trans["specialty_id"].astype(int).tolist()
le = LabelEncoder()
y = le.fit_transform(df_trans["specialty_id"].astype(int))

# **GPT transformer**

**preparing the data for GPT**

In [ ]:
#initializing gpt2-small-arabic Auto toknizer using("gpt2-small-arabic")
gpt_tokenizer = AutoTokenizer.from_pretrained("akhooli/gpt2-small-arabic")
# GPT2 dosent have built in padding that is why this step is necessary
gpt_tokenizer.pad_token = gpt_tokenizer.eos_token
#here we are transforming the data to a form that gpt understands where we use padding to ensure that all the inputs have the same length
#ensuring that all texts are shorter than 130 tokens
#and lastly ensuring that the output works with tensorflow
encodings_tokenization_gpt = gpt_tokenizer(x, truncation=True, padding="max_length", max_length=130, return_tensors="tf")
#attention mask that helps the model now the difference between padding tokens and actual tokens
attention_mask_gpt = encodings_tokenization_gpt["attention_mask"].numpy()
#input formed as toknized numpy array
input_gpt = encodings_tokenization_gpt["input_ids"].numpy()
#the target as numpy array
target_gpt = np.array(y)
#splitting and preparing testing and training data
#%80 training rest testing
X_train_in_gpt, X_test_in_gpt, X_train_attention_gpt, X_test_attention_gpt, y_train_gpt, y_test_gpt = train_test_split(input_gpt, attention_mask_gpt, target_gpt, test_size=0.2, random_state=42)
# Train and test datasets using tensor slices where batch = 8
train_gpt = tf.data.Dataset.from_tensor_slices(({"input_ids": X_train_in_gpt, "attention_mask": X_train_attention_gpt},y_train_gpt)).batch(8)
test_gpt = tf.data.Dataset.from_tensor_slices(({"input_ids": X_test_in_gpt, "attention_mask": X_test_attention_gpt},y_test_gpt)).batch(8)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/30.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/666 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.55M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.21M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/120 [00:00<?, ?B/s]

In [ ]:
#Initial learning rate = init_lr=3e-5
#	num_train_steps = number of batches(8) × epochs(4)
optimizer_gpt, schedule_gpt = create_optimizer(init_lr=3e-5,num_warmup_steps=int(0.1*len(train_gpt) * 4),num_train_steps=len(train_gpt) * 4)

In [ ]:
#preparing the gpt2-small-arabic model
#TFAutoModelForSequenceClassification is a tenserflow version of gpt that is spicifcally for classification tasks
model_gpt = TFAutoModelForSequenceClassification.from_pretrained("akhooli/gpt2-small-arabic",num_labels=len(le.classes_),from_pt=True)

pytorch_model.bin:   0%|          | 0.00/510M [00:00<?, ?B/s]

Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFGPT2ForSequenceClassification: ['transformer.h.5.attn.masked_bias', 'transformer.h.8.attn.masked_bias', 'transformer.h.3.attn.masked_bias', 'transformer.h.6.attn.masked_bias', 'transformer.h.0.attn.masked_bias', 'transformer.h.9.attn.masked_bias', 'transformer.h.11.attn.masked_bias', 'transformer.h.10.attn.masked_bias', 'transformer.h.7.attn.masked_bias', 'transformer.h.4.attn.masked_bias', 'transformer.h.1.attn.masked_bias', 'lm_head.weight', 'transformer.h.2.attn.masked_bias']
- This IS expected if you are initializing TFGPT2ForSequenceClassification from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFGPT2ForSequenceClassification from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassificat

In [ ]:
#compiling the bert model using the optimizer we initialized and setting the loss function to SparseCategoricalCrossentropy which is user for classification task
model_gpt.compile(optimizer=optimizer_gpt, loss= tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True), metrics=["accuracy"])

In [ ]:
gpt_training = model_gpt.fit(train_gpt, validation_data=test_gpt, epochs=4)

Epoch 1/4
2500/2500 [==============================] - 659s 250ms/step - loss: 0.7656 - accuracy: 0.7120 - val_loss: 0.4277 - val_accuracy: 0.8530
Epoch 2/4
2500/2500 [==============================] - 627s 251ms/step - loss: 0.3548 - accuracy: 0.8761 - val_loss: 0.3846 - val_accuracy: 0.8722
Epoch 3/4
2500/2500 [==============================] - 627s 251ms/step - loss: 0.2630 - accuracy: 0.9091 - val_loss: 0.3787 - val_accuracy: 0.8766
Epoch 4/4
2500/2500 [==============================] - 626s 250ms/step - loss: 0.2127 - accuracy: 0.9284 - val_loss: 0.3865 - val_accuracy: 0.8776


In [ ]:
y_actual = []
y_predicted_gpt= []
#going through each batch
for b in test_gpt:
#unpaking inputs and target
    x, y = b
    #the model making predictions
    outputs = model_gpt(x, training=False)
    #logits are raw scores before applying softmax, where each with its corusponding class and sample
    logits = outputs.logits
    #this returns the maximum logit which shows which class is the predicted
    predictions = tf.argmax(logits, axis=1)
    #convert to to numpy
    y_actual.extend(y.numpy())
    #adding the batch values to the list
    y_predicted_gpt.extend(predictions.numpy())

In [ ]:
print(classification_report(y_actual, y_predicted_gpt, target_names=le.classes_.astype(str)))

              precision    recall  f1-score   support

          14       0.88      0.89      0.89      1357
          18       0.91      0.92      0.91      1020
          23       0.89      0.89      0.89       810
          25       0.88      0.85      0.86       782
          91       0.82      0.84      0.83      1031

    accuracy                           0.88      5000
   macro avg       0.88      0.88      0.88      5000
weighted avg       0.88      0.88      0.88      5000

